# Notebook 04: CIFAR-10 CNN analysis

Full analysis of the NMF backward factor trace on CIFAR-10 (SmallCNN), keeping only the
two hardware-justified simplifications (stimulus filter + conv spatial pooling).

Deviations from NB02 (MNIST MLP baseline) are explained in Section 8.

In [ ]:
import os

# ── Paths (repo-local) ────────────────────────────────────────────────────────
_nb_dir   = os.path.dirname(os.path.abspath('__file__'))   # notebooks/
_repo_dir = os.path.dirname(_nb_dir)                       # repo root

CACHE_DIR    = os.path.join(_repo_dir, 'data', 'cache', 'nb04')
DATA_DIR     = os.path.join(_repo_dir, 'data')
WEIGHTS_PATH = os.path.join(CACHE_DIR, 'weights', 'weights.pt')

FIG_DIR  = os.path.join(_repo_dir, 'figs', '04_cifar10_trace')
PATH_DIR = os.path.join(CACHE_DIR, 'paths')
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(PATH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ── Stimulus selection ────────────────────────────────────────────────────────
FILTER_METHOD   = 'confidence'   # 'confidence' | 'random' | 'none'
N_TOP_PER_CLASS = 60

# ── Conv spatial pooling ──────────────────────────────────────────────────────
POOL_METHOD = 'avg'   # 'avg' | 'max' | 'center'

# ── NMF rank configuration ────────────────────────────────────────────────────
# Per-layer lists (forward order): conv1, conv2, conv3, fc1, fc2
K_MAX_LIST   = [6, 6, 6, 6, 12]
K_FIXED_LIST = [None, None, None, None, None]

# ── Branching ─────────────────────────────────────────────────────────────────
N_BRANCHES = [1, 1, 1, 2, 5]

# ── Caching ───────────────────────────────────────────────────────────────────
USE_CACHED_NMF = False

# ── Scaffold graph ────────────────────────────────────────────────────────────
SCAFFOLD_EDGE_THRESHOLD = 0.85

# ── Faithfulness evaluation ───────────────────────────────────────────────────
ABLATION_FRACTIONS = [0.02, 0.05, 0.10, 0.20]

RNG_SEED = 42
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

import matplotlib
matplotlib.rcParams.update({'figure.dpi': 80})

print(f'CACHE_DIR={CACHE_DIR}')
print(f'DATA_DIR={DATA_DIR}')
print(f'WEIGHTS_PATH={WEIGHTS_PATH}')
print(f'FIG_DIR={FIG_DIR}')
print(f'FILTER_METHOD={FILTER_METHOD}  N_TOP_PER_CLASS={N_TOP_PER_CLASS}')
print(f'POOL_METHOD={POOL_METHOD}')
print(f'K_MAX_LIST={K_MAX_LIST}  K_FIXED_LIST={K_FIXED_LIST}')
print(f'N_BRANCHES={N_BRANCHES}  USE_CACHED_NMF={USE_CACHED_NMF}')

In [ ]:
import sys, json, time, pickle, warnings, copy
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors
import networkx as nx
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
sys.path.insert(0, '..')

from src import (
    run_nmf, normalize_factors, sort_by_lambda, full_nmf_pipeline, auto_nmf_pipeline,
    bft, compute_conv_joint_arbors, compute_joint_arbors_normalized,
    collect_layer_data, plot_scaffold_graph,
)
print('src imports OK')

In [ ]:
# Arbor computation (compute_conv_joint_arbors, compute_joint_arbors_normalized)
# is now provided by src/bft.py and imported in the cell above.
print('Arbor functions: using src/bft.py implementations.')

In [ ]:
# collect_layer_data is imported from src (cell above).

def confidence_prefilter(raw, top_k, n_classes=10):
    """Keep top_k most-confident samples per class."""
    keep = np.sort(np.concatenate([
        np.where(raw['targets'] == c)[0][
            np.argsort(raw['confidences'][raw['targets'] == c])[::-1][:top_k]]
        for c in range(n_classes)]))
    return {k: (v[keep] if isinstance(v, np.ndarray) else
                [{**ld, 'input_fmap': ld['input_fmap'][keep],
                  'output_fmap': ld['output_fmap'][keep]} for ld in v])
            for k, v in raw.items()}, keep


def random_filter(raw, top_k, seed=42, n_classes=10):
    """Keep a random top_k samples per class."""
    rng  = np.random.RandomState(seed)
    keep = np.sort(np.concatenate([
        rng.choice(np.where(raw['targets'] == c)[0],
                   min(top_k, (raw['targets'] == c).sum()), replace=False)
        for c in range(n_classes)]))
    return {k: (v[keep] if isinstance(v, np.ndarray) else
                [{**ld, 'input_fmap': ld['input_fmap'][keep],
                  'output_fmap': ld['output_fmap'][keep]} for ld in v])
            for k, v in raw.items()}, keep


# -- Caching

def save_obj(obj, path):
    with open(path, 'wb') as f: pickle.dump(obj, f, protocol=4)

def load_obj(path):
    with open(path, 'rb') as f: return pickle.load(f)

def save_arrays(path, **arrays):
    np.savez_compressed(path, **arrays)

def load_arrays(path):
    return dict(np.load(path, allow_pickle=True))


# -- Memory

def print_mem(label=''):
    try:
        import psutil
        p    = psutil.Process()
        rss  = p.memory_info().rss / 1e9
        avail = psutil.virtual_memory().available / 1e9
        total = psutil.virtual_memory().total / 1e9
        print(f'[RAM {label}] rss={rss:.2f}GB  avail={avail:.2f}GB  total={total:.2f}GB')
    except ImportError:
        print('[RAM] psutil not available -- run: pip install psutil')


# -- Validation helpers

def top_cosine_sim(neu_f_a, neu_f_b):
    """Cosine similarity between top columns of two neural_factors matrices,
    after Hungarian matching (handles permutation)."""
    K = min(neu_f_a.shape[1], neu_f_b.shape[1])
    A = neu_f_a[:, :K]; B = neu_f_b[:, :K]
    A = A / (np.linalg.norm(A, axis=0, keepdims=True) + 1e-12)
    B = B / (np.linalg.norm(B, axis=0, keepdims=True) + 1e-12)
    C = A.T @ B   # (K, K) cosine similarities
    row, col = linear_sum_assignment(-np.abs(C))
    return np.abs(C[row, col]).mean()


def img_factor_cosine(if_a, if_b, col=0):
    """Cosine similarity between img_factors[:, col] of two runs."""
    a = if_a[:, col]; b = if_b[:, col]
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(a @ b)


def imdenorm(img, mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)):
    m = np.array(mean)[None, None, :]; s = np.array(std)[None, None, :]
    return np.clip(img.transpose(1, 2, 0) * s + m, 0, 1)

print('Utilities loaded.')

In [ ]:
def sparsify_edge_matrix(E, threshold_percentile):
    """Zero entries of E below the given percentile of abs values (per matrix)."""
    if threshold_percentile <= 0 or E.max() == 0:
        return E
    cutoff = np.percentile(np.abs(E[E != 0]), threshold_percentile * 100) if (E != 0).any() else 0
    return np.where(np.abs(E) >= cutoff, E, 0.0)

# plot_scaffold_graph is imported from src (cell above).
print('Scaffold helpers loaded.')

## Section 2: Remaining Simplifications

NB07 retains exactly two simplifications from NB06. All others (stimulus threshold,
arbor compression, fixed K, conv branching) are removed.

| Simplification | Flag | Status |
|---|---|---|
| S1: Stimulus filter | `FILTER_METHOD`, `N_TOP_PER_CLASS` | **Kept** -- user-selectable |
| S2: Spatial pooling | `POOL_METHOD` | **Kept** -- required for conv layers |
| S3: Stimulus threshold | -- | Removed (always 0.0) |
| S4: Arbor compression | -- | Removed (always full arbor) |
| S5: Fixed K | `K_FIXED_LIST` | Kept as optional override; default auto |
| S6: Conv branching | `N_BRANCHES` | Per-layer control list |


### 2a -- Stimulus Filter (S1)

**What it does:** After collecting all correctly-classified test samples, we keep only
`N_TOP_PER_CLASS` samples per class.

- `'confidence'`: sort by max-softmax confidence (highest first). Keeps the samples the
  network is most certain about. Biases toward "prototypical" examples for each class.

- `'random'`: draw a random subset of size `N_TOP_PER_CLASS` per class. No bias toward easy examples.

- `'none'`: use all correctly-classified samples (~900/class for CIFAR-10 test set).

**Why we filter:** The NMF input matrix is `(N x D)` where `D` = arbor dimension.
At N=60/class (600 total), the fc1 arbor is ~30 MB. At N=9000 (all correct), it would
be ~4.5 GB -- infeasible on most hardware. Filtering also makes NMF faster and more stable.

**Effect on factors:** Top-confidence samples are highly class-consistent, so factors
tend to be sharper and more class-specific. Random samples add diversity.
The confidence distribution plot below shows where the cut-off falls per class.


### 2b -- Conv Spatial Pooling (S2)

**Why pooling is necessary:** For a conv layer, the "synaptic arbor" is the outer product
of every weight connection with every input patch. Without pooling, conv1 alone would
produce a `(N, C_out x C_in x kH x kW x H x W)` = `(N, 16x3x3x3 x 32x32)` = `(N, 442,368)` matrix --
~1,000x too large for NMF.

**How it works -- step by step with SmallCNN dimensions:**

```
Step 1 - Extract weight tensor
  conv1: weight (C_out=16, C_in=3, kH=3, kW=3)
  conv2: weight (C_out=32, C_in=16, kH=3, kW=3)
  conv3: weight (C_out=64, C_in=32, kH=3, kW=3)

Step 2 - Unfold input feature map into sliding-window patches
  conv1: input_fmap (N, 3, 32, 32)
         F.unfold(..., kernel=(3,3), padding=(1,1))
         -> patches (N, C_in*kH*kW=27, n_pos=1024)  [32x32 positions]

  conv2: input_fmap (N, 16, 16, 16)   [after MaxPool2d(2)]
         -> patches (N, 144, 256)                     [16x16 positions]

  conv3: input_fmap (N, 32, 8, 8)     [after MaxPool2d(2)]
         -> patches (N, 288, 64)                      [8x8 positions]

Step 3 - Reduce over spatial positions (choose one):
  avg:    patches.mean(dim=2)           -> (N, C_in*kH*kW)
  max:    patches.max(dim=2).values     -> (N, C_in*kH*kW)
  center: patches[:, :, H//2*W + W//2] -> (N, C_in*kH*kW)

Step 4 - Weight x pooled-input outer product
  W_flat:  weight.reshape(C_out, C_in*kH*kW)
  joint:   (pooled[:, None, :] * W_flat[None, :, :])
           .reshape(N, C_out * C_in*kH*kW)

  conv1 joint: (N,  16 *  27) = (N,   432)   [vs full: 442,368]
  conv2 joint: (N,  32 * 144) = (N, 4,608)
  conv3 joint: (N,  64 * 288) = (N, 18,432)

Step 5 - Clip to [0, inf) and run NMF
  pos_joint = np.clip(joint, 0, None)   # non-negative for NMF
  W, H, lams = nmf_pipeline(pos_joint, K)
  # W = img_factors (N, K)  -- sample-level coefficients
  # H = neural_factors (C_out*C_in*kH*kW, K)  -- weight-space patterns
```

**Pooling method comparison:**
- **avg** (default): global average over all positions -- captures what a filter responds to
  *on average* across the image. Best for texture-like patterns spread over the image.
- **max**: most activated position per kernel entry -- captures peak activations, better for
  localized features (edges, corners). Can amplify noise.
- **center**: single center patch only -- fastest, but ignores border structure. Good sanity check.


In [ ]:
# -- SmallCNN (must match NB04/NB06 checkpoint)

class SmallCNN(nn.Module):
    """
    3-conv + 2-FC network for CIFAR-10.
    conv1 (3->16)  -> BN -> ReLU -> MaxPool2d(2)       input: (N,3,32,32)
    conv2 (16->32) -> BN -> ReLU -> MaxPool2d(2)       input: (N,16,16,16)
    conv3 (32->64) -> BN -> ReLU -> AdaptiveAvgPool(4) input: (N,32,8,8)
    fc1   (1024->128)  -> ReLU                         input: (N,1024)
    fc2   (128->10)    -> log_softmax                  input: (N,128)
    """
    def __init__(self):
        super().__init__()
        self.conv1  = nn.Conv2d(3,  8, 3, padding=1)
        self.bn1    = nn.BatchNorm2d(8)
        self.conv2  = nn.Conv2d(8, 16, 3, padding=1)
        self.bn2    = nn.BatchNorm2d(16)
        self.conv3  = nn.Conv2d(16, 32, 3, padding=1)
        self.bn3    = nn.BatchNorm2d(32)
        self.pool   = nn.MaxPool2d(2)
        self.avgpool = nn.AdaptiveAvgPool2d(4)
        self.fc1    = nn.Linear(32 * 4 * 4, 32)
        self.fc2    = nn.Linear(32, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.avgpool(F.relu(self.bn3(self.conv3(x))))
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)


In [ ]:
model = SmallCNN().to(DEVICE)

if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    print(f'Loaded weights from: {WEIGHTS_PATH}')
else:
    _save_dir = os.path.dirname(WEIGHTS_PATH)
    os.makedirs(_save_dir, exist_ok=True)
    print(f'Weights not found -- training from scratch (~5 min on GPU, ~20 min on CPU)')
    N_EPOCHS  = 150
    _normalize = T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
    _train_tf = T.Compose([T.RandomCrop(32,4),T.RandomHorizontalFlip(),T.ToTensor(),_normalize])
    _test_tf  = T.Compose([T.ToTensor(), _normalize])
    _train_ds = torchvision.datasets.CIFAR10(DATA_DIR,True, download=True,transform=_train_tf)
    _test_ds  = torchvision.datasets.CIFAR10(DATA_DIR,False,download=True,transform=_test_tf)
    _tr = torch.utils.data.DataLoader(_train_ds,128,shuffle=True, num_workers=2,pin_memory=True)
    _te = torch.utils.data.DataLoader(_test_ds, 256,shuffle=False,num_workers=2,pin_memory=True)
    opt  = torch.optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
    crit = nn.NLLLoss()
    for ep in range(1, N_EPOCHS+1):
        model.train()
        nc, nt, rl = 0, 0, 0.0
        for x, y in _tr:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
            out=model(x); loss=crit(out,y); loss.backward(); opt.step()
            nc+=(out.argmax(1)==y).sum().item(); nt+=y.size(0); rl+=loss.item()*y.size(0)
        sch.step()
        if ep%10==0 or ep==N_EPOCHS:
            model.eval()
            tc=ts=0
            with torch.no_grad():
                for x,y in _te:
                    x,y=x.to(DEVICE),y.to(DEVICE)
                    tc+=(model(x).argmax(1)==y).sum().item(); ts+=y.size(0)
            print(f'Ep {ep:2d} | train {nc/nt:.3f} | test {tc/ts:.3f}')
    torch.save(model.state_dict(), WEIGHTS_PATH)
    print(f'Saved to {WEIGHTS_PATH}')

# Per-class accuracy
normalize   = T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
test_tf     = T.Compose([T.ToTensor(), normalize])
test_ds     = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)
test_loader = torch.utils.data.DataLoader(test_ds, 256, shuffle=False, num_workers=0)

model.eval()
pc_correct = defaultdict(int); pc_total = defaultdict(int)
with torch.no_grad():
    for x, y in test_loader:
        x,y = x.to(DEVICE),y.to(DEVICE)
        preds = model(x).argmax(1)
        for t,p in zip(y.cpu().tolist(),preds.cpu().tolist()):
            pc_total[t]+=1; pc_correct[t]+=int(t==p)

print(f'{"Class":<14} {"Acc":>6}')
print('-'*22)
for c in range(10):
    print(f'{CIFAR10_CLASSES[c]:<14} {pc_correct[c]/pc_total[c]:>6.3f}')

## Section 4: Data Collection & Filtering

Collect (input, output) activations for every Conv2d and Linear layer via forward hooks.
Results cached to disk. After filtering, the full raw_data is deleted to free RAM.


In [ ]:
_raw_cache = os.path.join(CACHE_DIR, 'raw_data.pkl')

if os.path.exists(_raw_cache):
    print(f'Loading cached raw_data from {_raw_cache}')
    raw_data = load_obj(_raw_cache)
else:
    print('Collecting layer data from test set...')
    t0 = time.time()
    raw_data = collect_layer_data(model, test_loader, DEVICE)
    print(f'Collected {len(raw_data["targets"]):,} correct samples in {time.time()-t0:.1f}s')
    save_obj(raw_data, _raw_cache)
    print(f'Cached to {_raw_cache}')

print_mem('after raw_data load')
print()
print(f'{"Layer":<8}  {"Type":<4}  {"Input shape":<22}  {"W shape"}')
print('-'*60)
for ld in raw_data['layer_data']:
    wsh = str(ld['weight'].shape)
    ish = str(ld['input_fmap'].shape)
    print(f'{ld["name"]:<8}  {ld["type"]:<4}  {ish:<22}  {wsh}')

# -- Apply filter
_n_total = len(raw_data['targets'])

if FILTER_METHOD == 'confidence' and N_TOP_PER_CLASS is not None:
    data, _keep_idx = confidence_prefilter(raw_data, N_TOP_PER_CLASS)
    print(f'\nConfidence filter: {_n_total:,} -> {len(data["targets"])} samples '
          f'(top-{N_TOP_PER_CLASS}/class)')
elif FILTER_METHOD == 'random' and N_TOP_PER_CLASS is not None:
    data, _keep_idx = random_filter(raw_data, N_TOP_PER_CLASS, seed=RNG_SEED)
    print(f'\nRandom filter: {_n_total:,} -> {len(data["targets"])} samples '
          f'(random-{N_TOP_PER_CLASS}/class)')
else:
    data     = raw_data
    _keep_idx = np.arange(_n_total)
    print(f'\nNo filter: using all {_n_total:,} correct samples')

print_mem('after filter')
N = len(data['targets'])
targets = data['targets']
images  = data['images']
print(f'N = {N} samples across {len(data["layer_data"])} layers')

# -- Confidence distribution plot
fig, ax = plt.subplots(figsize=(10, 3.5))
rng = np.random.default_rng(0)
for c in range(10):
    mask_all  = raw_data['targets'] == c
    all_confs = raw_data['confidences'][mask_all]
    jitter    = rng.uniform(-0.08, 0.08, mask_all.sum())
    ax.scatter(all_confs, np.full(mask_all.sum(), c) + jitter,
               s=3, alpha=0.25, c=f'C{c}')
    if FILTER_METHOD == 'confidence' and N_TOP_PER_CLASS is not None:
        thresh_val = np.sort(all_confs)[::-1][N_TOP_PER_CLASS - 1]
        ax.axvline(thresh_val, color=f'C{c}', lw=1.2, alpha=0.7)
ax.set_yticks(range(10))
ax.set_yticklabels(CIFAR10_CLASSES)
ax.set_xlabel('Max softmax confidence')
_filter_desc = (f'top-{N_TOP_PER_CLASS}/class kept (vertical = threshold)'
                if FILTER_METHOD == 'confidence' else
                f'random-{N_TOP_PER_CLASS}/class' if FILTER_METHOD == 'random' else
                'all correct samples (no filter)')
ax.set_title(f'Confidence distribution -- {_filter_desc}')
plt.tight_layout()
plt.show()

# Free raw_data if filtered
if FILTER_METHOD in ('confidence', 'random') and N_TOP_PER_CLASS is not None:
    del raw_data
    print('raw_data freed from RAM.')


## Section 5: Backward NMF Factor Tree Trace

Runs NMF backward from fc2 (output) to conv1 (input) using `bft` from
`src/bft.py`. At each layer:
1. Compute the synaptic arbor (weight × pooled-input)
2. Run NMF at `K_MAX` (or fixed K); auto-K is selected by the `structural`
   method (fraction ratio as primary signal, cumvar floor at 95%)
3. Use the top factor's img_factors as stimulus weights for the next (shallower) layer

Per-path caching: when `USE_CACHED_NMF=True`, `cache_dir=PATH_DIR` is passed to
`bft`, which saves completed leaf paths to `PATH_DIR/bft_path_<seq>.pkl` and
reloads them on the next run. The full assembled tree is also saved to
`CACHE_DIR/tree.pkl` for fast re-analysis.

In [ ]:
# Layer NMF configuration and tracing are now handled by bft (src/bft.py).
# K_MAX_LIST and K_FIXED_LIST are passed directly to bft() in the next cell.
print(f'BFT configuration: K_MAX_LIST={K_MAX_LIST}  K_FIXED_LIST={K_FIXED_LIST}')

In [ ]:
def get_spine(root):
    nodes, node = [], root
    while node:
        nodes.append(node)
        node = node['children'][0] if node['children'] else None
    return nodes


def get_all_paths(root):
    if not root['children']: return [[root]]
    return [[root] + sub for c in root['children'] for sub in get_all_paths(c)]

print('Tree helpers loaded.')

In [ ]:
_tree_cache = os.path.join(CACHE_DIR, 'tree.pkl')

if USE_CACHED_NMF and os.path.exists(_tree_cache):
    print(f'Loading cached tree from {_tree_cache}')
    tree = load_obj(_tree_cache)
else:
    t0 = time.time()
    tree = bft(
        data['layer_data'],
        k_max             = K_MAX_LIST,
        k_fixed           = K_FIXED_LIST,
        n_branches        = N_BRANCHES,
        conv_pool_method  = POOL_METHOD,
        cache_dir         = PATH_DIR if USE_CACHED_NMF else None,
        stimulus_threshold = 0.0,
        random_state      = RNG_SEED,
        weighting         = 'img_selectivity',
        verbose           = 2
    )
    print(f'Done in {time.time()-t0:.1f}s')
    save_obj(tree, _tree_cache)
    print(f'Tree saved to {_tree_cache}')

print_mem('after trace')
spine = get_spine(tree)
paths = get_all_paths(tree)
print(f'\nSpine: {[n["layer_name"] for n in spine]}')
print(f'Paths: {len(paths)}')
for pi, path in enumerate(paths):
    print(f'  Path {pi}: ' + ' -> '.join(
        f'{n["layer_name"]}[{n["factor_idx"]}]  K={len(n["lambdas"])}  act={n["active_samples"]}'
        for n in path))

## Section 6: Full Visualization Suite

Six visualization sections derived from NB04, showing different aspects of the discovered
factor tree. All figures are also saved to `FIG_DIR`.


## Section 6a: Per-Layer Factor Summaries

For each layer on each path: (1) lambda bar chart, (2) per-class mean stimulus weight,
(3) weighted average image, (4) neural factor activation map.


In [ ]:
def imdenorm(img_np,
             mean=(0.4914, 0.4822, 0.4465),
             std=(0.2470, 0.2435, 0.2616)):
    m = np.array(mean)[None, None, :]
    s = np.array(std)[None, None, :]
    return np.clip(img_np.transpose(1, 2, 0) * s + m, 0, 1)


def _act_map(node, k, layer_data_list):
    neu_k = node['neural_factors'][:, k]
    weight = layer_data_list[node['layer_idx']]['weight']

    if node['layer_type'] == 'fc':
        n_out, n_in = weight.shape
        return neu_k.reshape(n_out, n_in), 'input neuron', 'output neuron'
    else:  # conv
        C_out, C_in, kH, kW = weight.shape
        return (neu_k.reshape(C_out, C_in * kH * kW),
                f'C_in*kernel  (C_in={C_in}, k={kH}x{kW})', 'C_out')


layer_data_list = data['layer_data']
_inline_shown_6a = 0
_MAX_INLINE_6a = 3  # show at most 3 path summaries inline; rest saved only

for pi, path in enumerate(paths):
    path_label = ' -> '.join(
        f"{n['layer_name']}[{n['factor_idx']}]" for n in path)

    layer_Ks   = [len(n['lambdas']) for n in path]
    total_rows = sum(layer_Ks)
    row_height = 1.8
    fig, axes = plt.subplots(total_rows, 4,
                             figsize=(16, row_height * total_rows),
                             squeeze=False)

    row = 0
    for ni, node in enumerate(path):
        lams   = node['lambdas']
        K_node = len(lams)

        for k in range(K_node):
            stim_k = node['img_factors'][:, k]

            ax0 = axes[row, 0]
            bar_colors = ['#c0392b' if i == k else '#bdc3c7'
                          for i in range(K_node)]
            ax0.bar(range(K_node), lams, color=bar_colors, edgecolor='white')
            ax0.set_xlabel('factor', fontsize=6)
            ax0.set_ylabel('lambda', fontsize=6)
            ax0.tick_params(labelsize=6)
            title_top = (f'{node["layer_name"]}  ({node["layer_type"]})  '
                         f'active={node["active_samples"]}')
            ax0.set_title(title_top if k == 0 else f'factor {k}  lambda={lams[k]:.2f}',
                          fontsize=7, fontweight='bold' if k == 0 else 'normal')

            ax1 = axes[row, 1]
            means = [stim_k[targets == c].mean() for c in range(10)]
            ax1.bar(range(10), means,
                    color=[f'C{c}' for c in range(10)], alpha=0.85)
            ax1.set_xticks(range(10))
            ax1.set_xticklabels(CIFAR10_CLASSES, rotation=45,
                                ha='right', fontsize=5)
            ax1.set_ylabel('mean stim weight', fontsize=6)
            ax1.set_title(f'class weights  k={k}  lambda={lams[k]:.2f}', fontsize=7)
            ax1.tick_params(labelsize=6)

            ax2 = axes[row, 2]
            w   = np.maximum(stim_k, 0)
            w  /= w.sum() + 1e-8
            avg = (images * w[:, None, None, None]).sum(0)
            ax2.imshow(imdenorm(avg))
            ax2.axis('off')
            ax2.set_title(f'weighted avg image  k={k}', fontsize=7)

            ax3 = axes[row, 3]
            act_map, xlabel, ylabel = _act_map(node, k, layer_data_list)
            im = ax3.imshow(act_map, aspect='auto', cmap='PuRd')
            plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
            ax3.set_xlabel(xlabel, fontsize=6)
            ax3.set_ylabel(ylabel, fontsize=6)
            ax3.tick_params(labelsize=5)
            ax3.set_title(f'activation map  k={k}', fontsize=7)

            row += 1

        if ni < len(path) - 1:
            sep_row = row - 1
            for col in range(4):
                axes[sep_row, col].spines['bottom'].set(
                    linewidth=2.5, color='#555555')

    fig.suptitle(f'Path {pi}:  {path_label}', fontsize=11, y=1.002)
    plt.tight_layout()
    fname = os.path.join(FIG_DIR, f'factor_summary_path{pi}.png')
    fig.savefig(fname, dpi=100, bbox_inches='tight')
    if _inline_shown_6a < _MAX_INLINE_6a:
        plt.show()
        _inline_shown_6a += 1
    else:
        plt.close(fig)
    print(f'  Saved to {fname}')

## Section 6b: Conv Factor Kernel Visualization

For each conv layer on the spine: show the top-8 output channels by NMF loading,
rendered as RGB patches (conv1) or weight matrices (conv2/conv3).


In [ ]:
N_SHOW_CHANNELS = 8

seen_nodes = set()
_inline_shown_6b = 0
_MAX_INLINE_6b = 5  # max 5 kernel viz figures inline

for path_idx, path in enumerate(paths):
    for node in path:
        if node['layer_type'] != 'conv':
            continue
        node_id = id(node)
        if node_id in seen_nodes:
            continue
        seen_nodes.add(node_id)

        ld    = data['layer_data'][node['layer_idx']]
        C_out, C_in, kH, kW = ld['weight'].shape
        neu_f = node['neural_factors']   # (D, K)

        path_str = '/'.join(str(b) for b in node['path'])

        top_f       = neu_f[:, 0].reshape(C_out, C_in, kH, kW)
        ch_load     = top_f.reshape(C_out, -1).sum(1)
        top_ch_idx  = np.argsort(ch_load)[::-1][:N_SHOW_CHANNELS]
        n_show      = len(top_ch_idx)

        if C_in == 3:
            fig, axes = plt.subplots(1, n_show, figsize=(2.2 * n_show, 2.5))
            if n_show == 1: axes = [axes]
            for col, ch in enumerate(top_ch_idx):
                k   = top_f[ch].transpose(1, 2, 0)
                k   = (k - k.min()) / (k.max() - k.min() + 1e-8)
                axes[col].imshow(k)
                axes[col].set_title(f'ch{ch}\n{ch_load[ch]:.2f}', fontsize=8)
                axes[col].axis('off')
            fig.suptitle(f'{node["layer_name"]} [path {path_str}]: top-{n_show} output channel kernels '
                         f'(top NMF factor, RGB)', fontsize=10)
        else:
            fig, axes = plt.subplots(1, n_show, figsize=(2.2 * n_show, 2.8))
            if n_show == 1: axes = [axes]
            for col, ch in enumerate(top_ch_idx):
                k     = top_f[ch].reshape(C_in * kH, kW)
                vmax  = np.abs(k).max()
                axes[col].imshow(k, cmap='RdBu_r', aspect='auto',
                                  vmin=-vmax, vmax=vmax)
                axes[col].set_title(f'ch{ch}\n{ch_load[ch]:.2f}', fontsize=8)
                axes[col].axis('off')
            fig.suptitle(f'{node["layer_name"]} [path {path_str}]: top-{n_show} output channels '
                         f'(C_in={C_in}, top NMF factor)', fontsize=10)

        plt.tight_layout()
        fname = f'{node["layer_name"]}_path{path_str.replace("/","-")}_kernels.png'
        fig.savefig(os.path.join(FIG_DIR, fname), dpi=120, bbox_inches='tight')
        if _inline_shown_6b < _MAX_INLINE_6b:
            plt.show()
            _inline_shown_6b += 1
        else:
            plt.close(fig)
        print(f'  {node["layer_name"]} [path {path_str}]: top lambda={node["lambdas"][0]:.3f} | '
              f'neural_factors {neu_f.shape}')

## Section 6c: Spatial Activation Maps

Re-run the top-factor's highest-weight samples through the model to get channel-weighted
spatial activation maps at each conv layer. Shows top-6 and bottom-6 samples side by side.


In [ ]:
def get_spatial_activation_map(model, images_np, node, layer_data, device,
                                n_images=6, which='top'):
    scores = node['img_factors'][:, 0]
    if which == 'top':
        sel_idx = np.argsort(scores)[::-1][:n_images]
    else:
        sel_idx = np.argsort(scores)[:n_images]

    ld    = layer_data[node['layer_idx']]
    C_out = ld['weight'].shape[0]
    neu_f = node['neural_factors']

    kH, kW = ld['weight'].shape[2], ld['weight'].shape[3]
    Ci     = ld['weight'].shape[1]
    ch_importance = neu_f[:, 0].reshape(C_out, Ci * kH * kW).sum(1)

    ch_importance = np.maximum(ch_importance, 0)
    if ch_importance.sum() > 0:
        ch_importance /= ch_importance.sum()

    feature_maps = {}
    target_mod   = dict(model.named_modules())[node['layer_name']]
    hook = target_mod.register_forward_hook(
        lambda mod, inp, out: feature_maps.update({'out': out.detach().cpu()}))

    model.eval()
    imgs_t = torch.from_numpy(images_np[sel_idx]).float().to(device)
    with torch.no_grad():
        model(imgs_t)
    hook.remove()

    fmap    = feature_maps['out'].numpy()
    spatial = np.maximum(
        (fmap * ch_importance[None, :, None, None]).sum(1), 0)
    return spatial, sel_idx


seen_nodes_6c    = set()
_inline_shown_6c = 0
_MAX_INLINE_6c   = 3

for path_idx, path in enumerate(paths):
    for node in path:
        if node['layer_type'] != 'conv':
            continue
        node_id = id(node)
        if node_id in seen_nodes_6c:
            continue
        seen_nodes_6c.add(node_id)

        path_str = '/'.join(str(b) for b in node['path'])

        top_maps, top_idx = get_spatial_activation_map(
            model, data['images'], node, data['layer_data'], DEVICE, n_images=6, which='top')
        bot_maps, bot_idx = get_spatial_activation_map(
            model, data['images'], node, data['layer_data'], DEVICE, n_images=6, which='bottom')

        n_show = len(top_idx)

        fig, axes = plt.subplots(4, n_show, figsize=(2.2 * n_show, 8))
        for col in range(n_show):
            axes[0, col].imshow(imdenorm(data['images'][top_idx[col]]))
            axes[0, col].set_title(CIFAR10_CLASSES[data['targets'][top_idx[col]]], fontsize=9)
            axes[0, col].axis('off')
            axes[1, col].imshow(top_maps[col], cmap='hot', aspect='auto')
            axes[1, col].axis('off')
            axes[2, col].imshow(imdenorm(data['images'][bot_idx[col]]))
            axes[2, col].set_title(CIFAR10_CLASSES[data['targets'][bot_idx[col]]], fontsize=9)
            axes[2, col].axis('off')
            axes[3, col].imshow(bot_maps[col], cmap='hot', aspect='auto')
            axes[3, col].axis('off')

        axes[0, 0].set_ylabel('Top\nimage', fontsize=9, labelpad=4)
        axes[1, 0].set_ylabel('Top\nmap', fontsize=9, labelpad=4)
        axes[2, 0].set_ylabel('Bottom\nimage', fontsize=9, labelpad=4)
        axes[3, 0].set_ylabel('Bottom\nmap', fontsize=9, labelpad=4)
        fig.suptitle(f'{node["layer_name"]} [path {path_str}]: top-factor spatial activation maps '
                     f'(top-{n_show} / bottom-{n_show} by stimulus weight)', fontsize=10)
        plt.tight_layout()
        fname = f'{node["layer_name"]}_path{path_str.replace("/","-")}_spatial.png'
        fig.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches='tight')
        if _inline_shown_6c < _MAX_INLINE_6c:
            _inline_shown_6c += 1
            plt.show()
        else:
            plt.close(fig)


## Section 6d: Class x Path Heatmap

Which CIFAR-10 classes dominate each leaf path? Uses the path-specific fc1 stimulus weights.


In [ ]:
n_paths     = len(paths)
heatmap     = np.zeros((10, n_paths))
path_labels = []

for pi, path in enumerate(paths):
    fc1_node   = next(n for n in path if n['layer_name'] == 'fc1')
    conv3_node = next(n for n in path if n['layer_name'] == 'conv3')
    fi_fc1     = min(conv3_node['factor_idx'], fc1_node['img_factors'].shape[1] - 1)
    stim       = fc1_node['img_factors'][:, fi_fc1]

    for c in range(10):
        heatmap[c, pi] = stim[targets == c].mean()

    branch_seq = path[-1]['path']
    path_labels.append('->'.join(str(b) for b in branch_seq) if branch_seq else 'main')

fig, ax = plt.subplots(figsize=(max(5, n_paths * 1.4), 4))
im = ax.imshow(heatmap, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_paths))
ax.set_xticklabels([f'Path {i}\n({l})' for i, l in enumerate(path_labels)], fontsize=9)
ax.set_yticks(range(10))
ax.set_yticklabels(CIFAR10_CLASSES)
ax.set_xlabel('Leaf path (full branch sequence)')
ax.set_ylabel('CIFAR-10 class')
ax.set_title('Mean fc1 stimulus weight per class x path\n'
             '(path-specific fc1 factor, not the shared fc2 root)')
plt.colorbar(im, ax=ax, label='Mean img_factor weight')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'class_path_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nDominant classes per pathway (fc1-level stimulus weights):')
for pi in range(n_paths):
    top3 = np.argsort(heatmap[:, pi])[::-1][:3]
    print(f'  Path {pi} ({path_labels[pi]}): ' +
          ' | '.join(f'{CIFAR10_CLASSES[c]} ({heatmap[c,pi]:.3f})' for c in top3))

print('\nNote -- paths sharing the same fc1 node (diverge only at conv layers):')
fc2_branch_map = defaultdict(list)
for pi, path in enumerate(paths):
    fc1_node = next(n for n in path if n['layer_name'] == 'fc1')
    fc2_branch_map[fc1_node['factor_idx']].append(pi)
for b, pis in fc2_branch_map.items():
    if len(pis) > 1:
        print(f'  fc2 branch {b}: paths {pis} share the same fc1 node')


## Section 6e: Full-Network Scaffold Graph

Extends NB04's FC-only scaffold graph to include **all layers** (conv + FC).
Nodes represent channels/neurons; edges represent NMF-factor-weighted connections.

**Layer sizes:** input (3 ch) -> conv1 (16 ch) -> conv2 (32 ch) -> conv3 (64 ch)
             -> fc1 (128) -> fc2 (10 classes)    Total: 253 nodes

**Edge matrices** (channel-aggregated):
- conv1 -> input:  neu_factors[:,k].reshape(16, 3, 3, 3).sum((-2,-1))   -> (16, 3)
- conv2 -> conv1:  neu_factors[:,k].reshape(32, 16, 3, 3).sum((-2,-1))  -> (32, 16)
- conv3 -> conv2:  neu_factors[:,k].reshape(64, 32, 3, 3).sum((-2,-1))  -> (64, 32)
- fc1 -> conv3:    neu_factors[:,k].reshape(128, 64, 4, 4).sum((-2,-1)) -> (128, 64)
                  (fc1 input = flattened conv3 output: 64 channels x 4x4 spatial)
- fc2 -> fc1:      neu_factors[:,k].reshape(10, 128)                     -> (10, 128)

`SCAFFOLD_EDGE_THRESHOLD` suppresses the weakest edges per boundary for readability.


In [ ]:
from src import build_scaffold_edges, scaffold_loading_from_edges, scaffold_layer_sizes_from_edges


def _fix_conv_fc_edges(edge_mats, neg_mats, path_shal, layer_data_list):
    for ni in range(1, len(path_shal)):
        node = path_shal[ni]
        prev_node = path_shal[ni - 1]
        if node['layer_type'] == 'fc' and prev_node['layer_type'] == 'conv':
            prev_C_out = layer_data_list[prev_node['layer_idx']]['weight'].shape[0]
            n_out, n_in = edge_mats[ni].shape
            H_sp = n_in // prev_C_out
            edge_mats[ni] = edge_mats[ni].reshape(n_out, prev_C_out, H_sp).sum(-1)
            if neg_mats[ni].any():
                neg_mats[ni] = neg_mats[ni].reshape(n_out, prev_C_out, H_sp).sum(-1)
            else:
                neg_mats[ni] = np.zeros((n_out, prev_C_out))


_inline_shown_6e = 0
_MAX_INLINE_6e   = 3

for pi, path in enumerate(paths):
    print(f'\n--- Path {pi} scaffold graph ---')
    path_shal = path[::-1]

    edge_mats, neg_mats = build_scaffold_edges(
        path_shal,
        fi='path',
        top_pct=0.05,
        aggregate_conv=True,
    )

    _fix_conv_fc_edges(edge_mats, neg_mats, path_shal, data['layer_data'])

    sparse_mats = [sparsify_edge_matrix(E, SCAFFOLD_EDGE_THRESHOLD) for E in edge_mats]

    layer_szs = scaffold_layer_sizes_from_edges(edge_mats)
    loading   = np.maximum(scaffold_loading_from_edges(edge_mats), 0)

    labels = [f'layer{i}({s})' for i, s in enumerate(layer_szs)]
    for i, (E, sE) in enumerate(zip(edge_mats, sparse_mats)):
        nnz_full   = (E   != 0).sum()
        nnz_sparse = (sE  != 0).sum()
        print(f'  {labels[i]}->{labels[i+1]}: {E.shape}  {nnz_full} edges -> {nnz_sparse} after threshold')

    fig = plot_scaffold_graph(loading, sparse_mats, layer_szs)
    branch_seq   = path[-1]['path']
    branch_label = '->'.join(str(b) for b in branch_seq) if branch_seq else 'main'
    fig.suptitle(f'Full-Network Scaffold Graph -- Path {pi} ({branch_label})\n'
                 f'Threshold={SCAFFOLD_EDGE_THRESHOLD:.0%} | Pool={POOL_METHOD}', fontsize=9)
    plt.tight_layout()
    fname = os.path.join(FIG_DIR, f'scaffold_path{pi}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    if _inline_shown_6e < _MAX_INLINE_6e:
        _inline_shown_6e += 1
        plt.show()
    else:
        plt.close(fig)
    print(f'  Saved to {fname}')


## Section 6f: Top-Factor Sample Gallery

For each path, show the top-16 samples by fc1 stimulus weight.


In [ ]:
N_GALLERY = 16

_inline_shown_6f = 0
_MAX_INLINE_6f   = 3

for pi, path in enumerate(paths):
    fc1_node   = next(n for n in path if n['layer_name'] == 'fc1')
    conv3_node = next(n for n in path if n['layer_name'] == 'conv3')
    fi_fc1     = min(conv3_node['factor_idx'], fc1_node['img_factors'].shape[1] - 1)
    stim       = fc1_node['img_factors'][:, fi_fc1]
    top_idx    = np.argsort(stim)[::-1][:N_GALLERY]

    ncols = 8
    nrows = (N_GALLERY + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.8, nrows * 2.0))
    axes = axes.flat

    for ax, idx in zip(axes, top_idx):
        ax.imshow(imdenorm(data['images'][idx]))
        ax.set_title(f'{CIFAR10_CLASSES[data["targets"][idx]]}\n'
                     f'w={stim[idx]:.3f}', fontsize=7)
        ax.axis('off')
    for ax in list(axes)[len(top_idx):]:
        ax.axis('off')

    branch_seq   = path[-1]['path']
    branch_label = '->'.join(str(b) for b in branch_seq) if branch_seq else 'main'
    fig.suptitle(f'Path {pi} ({branch_label}): top-{N_GALLERY} samples by fc1 stimulus weight',
                 fontsize=10)
    plt.tight_layout()
    fname = os.path.join(FIG_DIR, f'gallery_path{pi}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    if _inline_shown_6f < _MAX_INLINE_6f:
        _inline_shown_6f += 1
        plt.show()
    else:
        plt.close(fig)
    print(f'  Saved to {fname}')


## Section 7: Faithfulness Evaluation

**Question:** Do the NMF factors identify weights that actually control the computation,
or just aesthetically coherent patterns?

**Method (abbreviated NB03 ablation):** For each path, rank all network weights by their
NMF importance score. Ablate the top-p% most important weights (set to zero in a copy
of the model) and measure per-class test accuracy. Compare to random ablation of the
same number of weights.

**Expected result:** NMF ablation should damage the target class (the dominant class for
this path) more than random ablation, with less collateral damage to bystanders.
This class-specificity is the signature of a faithful computational circuit -- random
patterns would damage all classes equally.

**Specificity index:** `(NMF_target_drop / NMF_bystander_drop)` >
`(Random_target_drop / Random_bystander_drop)` means NMF is more class-targeted.


In [ ]:
def score_weights_by_nmf(path, layer_data_list):
    scores = {}
    for node_idx, node in enumerate(path):
        ld     = layer_data_list[node['layer_idx']]
        if node_idx == 0 and len(path) > 1:
            fi = path[1]['factor_idx']
        else:
            fi = 0
        fi     = min(fi, node['neural_factors'].shape[1] - 1)
        neu_k  = np.abs(node['neural_factors'][:, fi])
        weight = ld['weight']
        ltype  = node['layer_type']

        if ltype == 'conv':
            C_out, C_in, kH, kW = weight.shape
            score = neu_k.reshape(C_out, C_in, kH, kW)
        else:
            n_out, n_in = weight.shape
            score = neu_k.reshape(n_out, n_in)

        param_key = f'{ld["name"]}.weight'
        scores[param_key] = score
    return scores


def ablate_model(model, weight_scores, ablation_frac, seed=42):
    all_scores = np.concatenate([s.ravel() for s in weight_scores.values()])
    threshold  = np.quantile(all_scores, 1.0 - ablation_frac)

    state = copy.deepcopy(model.state_dict())
    for key, score in weight_scores.items():
        mask = torch.from_numpy((score >= threshold).astype(np.float32)).to(DEVICE)
        state[key] = state[key].to(DEVICE) * (1.0 - mask)

    ablated = SmallCNN().to(DEVICE)
    ablated.load_state_dict(state)
    return ablated


def ablate_model_random(model, weight_scores, ablation_frac, seed=42):
    rng = np.random.RandomState(seed)
    all_keys   = list(weight_scores.keys())
    all_shapes = [weight_scores[k].shape for k in all_keys]
    total_w    = sum(int(np.prod(s)) for s in all_shapes)
    n_ablate   = int(round(total_w * ablation_frac))

    flat_idx   = rng.permutation(total_w)[:n_ablate]
    flat_mask  = np.zeros(total_w, dtype=np.float32)
    flat_mask[flat_idx] = 1.0

    state = copy.deepcopy(model.state_dict())
    offset = 0
    for key, shape in zip(all_keys, all_shapes):
        n = int(np.prod(shape))
        mask = torch.from_numpy(flat_mask[offset:offset+n].reshape(shape)).to(DEVICE)
        state[key] = state[key].to(DEVICE) * (1.0 - mask)
        offset += n

    ablated = SmallCNN().to(DEVICE)
    ablated.load_state_dict(state)
    return ablated


def eval_per_class_acc(mdl, loader):
    mdl.eval()
    correct = np.zeros(10); total = np.zeros(10)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = mdl(x).argmax(1).cpu().numpy()
            yt    = y.cpu().numpy()
            for c in range(10):
                mask = yt == c
                correct[c] += (preds[mask] == c).sum()
                total[c]   += mask.sum()
    return correct / (total + 1e-12)


_normalize2  = T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
_test_tf2    = T.Compose([T.ToTensor(), _normalize2])
_test_ds2    = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=_test_tf2)
test_loader_full = torch.utils.data.DataLoader(_test_ds2, 256, shuffle=False, num_workers=0)

base_acc = eval_per_class_acc(model, test_loader_full)
print(f'Baseline accuracy: {base_acc.mean():.3f}')

all_nmf_target_drops    = {frac: [] for frac in ABLATION_FRACTIONS}
all_nmf_bystander_drops = {frac: [] for frac in ABLATION_FRACTIONS}
all_rnd_target_drops    = {frac: [] for frac in ABLATION_FRACTIONS}
all_rnd_bystander_drops = {frac: [] for frac in ABLATION_FRACTIONS}

_inline_shown_faith = 0
_MAX_INLINE_faith   = 3

for pi, path in enumerate(paths):
    fc1_node   = next(n for n in path if n['layer_name'] == 'fc1')
    conv3_node = next(n for n in path if n['layer_name'] == 'conv3')
    fi_fc1     = min(conv3_node['factor_idx'], fc1_node['img_factors'].shape[1]-1)
    stim       = fc1_node['img_factors'][:, fi_fc1]
    per_class_stim = np.array([stim[data['targets'] == c].mean() for c in range(10)])
    target_class   = int(np.argmax(per_class_stim))
    print(f'\nPath {pi}: dominant class = {CIFAR10_CLASSES[target_class]} ({target_class})')

    weight_scores = score_weights_by_nmf(path, data['layer_data'])

    nmf_target_drops    = []
    nmf_bystander_drops = []
    rnd_target_drops    = []
    rnd_bystander_drops = []

    for frac in ABLATION_FRACTIONS:
        mdl_nmf = ablate_model(model, weight_scores, frac, seed=RNG_SEED)
        acc_nmf = eval_per_class_acc(mdl_nmf, test_loader_full)
        nmf_target_drops.append(base_acc[target_class] - acc_nmf[target_class])
        bystander_mask = np.arange(10) != target_class
        nmf_bystander_drops.append((base_acc[bystander_mask] - acc_nmf[bystander_mask]).mean())

        mdl_rnd = ablate_model_random(model, weight_scores, frac, seed=RNG_SEED)
        acc_rnd = eval_per_class_acc(mdl_rnd, test_loader_full)
        rnd_target_drops.append(base_acc[target_class] - acc_rnd[target_class])
        rnd_bystander_drops.append((base_acc[bystander_mask] - acc_rnd[bystander_mask]).mean())

        print(f'  frac={frac:.0%}: NMF target d={nmf_target_drops[-1]:.3f}  '
              f'bystander d={nmf_bystander_drops[-1]:.3f}  |  '
              f'Rand target d={rnd_target_drops[-1]:.3f}  '
              f'bystander d={rnd_bystander_drops[-1]:.3f}')

    x_pos = np.arange(len(ABLATION_FRACTIONS))
    w   = 0.18
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x_pos - 1.5*w, nmf_target_drops,    w, label='NMF target',           color='#c0392b', alpha=0.9)
    ax.bar(x_pos - 0.5*w, nmf_bystander_drops, w, label='NMF bystander (mean)', color='#e74c3c', alpha=0.5)
    ax.bar(x_pos + 0.5*w, rnd_target_drops,    w, label='Random target',        color='#2c3e50', alpha=0.9)
    ax.bar(x_pos + 1.5*w, rnd_bystander_drops, w, label='Random bystander (mean)', color='#7f8c8d', alpha=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{f:.0%}' for f in ABLATION_FRACTIONS])
    ax.set_xlabel('Ablation fraction (top weights zeroed)')
    ax.set_ylabel('Accuracy drop from baseline')
    ax.legend(fontsize=8)

    nmf_spec = [td / (bd + 1e-6) for td, bd in zip(nmf_target_drops, nmf_bystander_drops)]
    rnd_spec = [td / (bd + 1e-6) for td, bd in zip(rnd_target_drops, rnd_bystander_drops)]
    spec_txt = '  |  '.join(f'{f:.0%}: NMF={s:.2f} vs Rand={r:.2f}'
                            for f, s, r in zip(ABLATION_FRACTIONS, nmf_spec, rnd_spec))
    ax.set_title(f'Path {pi} -- Target: {CIFAR10_CLASSES[target_class]}\n'
                 f'Specificity index (target_drop/bystander_drop): {spec_txt}', fontsize=8)
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f'faithfulness_path{pi}.png'), dpi=150, bbox_inches='tight')
    if _inline_shown_faith < _MAX_INLINE_faith:
        _inline_shown_faith += 1
        plt.show()
    else:
        plt.close(fig)

    for i, frac in enumerate(ABLATION_FRACTIONS):
        all_nmf_target_drops[frac].append(nmf_target_drops[i])
        all_nmf_bystander_drops[frac].append(nmf_bystander_drops[i])
        all_rnd_target_drops[frac].append(rnd_target_drops[i])
        all_rnd_bystander_drops[frac].append(rnd_bystander_drops[i])
    del mdl_nmf, mdl_rnd


In [ ]:
import scipy.stats as stats
import numpy as np

print("Statistical Testing: NMF Target Drops > Random Target Drops (across paths)")
print("-" * 80)

# We will test the hypothesis for each ablation fraction independently
for frac in ABLATION_FRACTIONS:
    nmf_drops = all_nmf_target_drops[frac]
    rnd_drops = all_rnd_target_drops[frac]
    
    # Paired t-test 
    # alternative='greater' checks if the mean of (nmf_drops - rnd_drops) > 0
    t_stat, p_val_t = stats.ttest_rel(nmf_drops, rnd_drops, alternative='greater')
    
    # Wilcoxon signed-rank test 
    # Non-parametric equivalent, robust to non-normal distributions in small sample sizes
    try:
        w_stat, p_val_w = stats.wilcoxon(nmf_drops, rnd_drops, alternative='greater')
    except ValueError: 
        # Handles cases where all differences are exactly zero (e.g., at 0% ablation)
        w_stat, p_val_w = float('nan'), float('nan')
        
    print(f"Ablation Fraction {frac:.0%}:")
    print(f"  Mean NMF Drop:  {np.mean(nmf_drops):.4f}")
    print(f"  Mean Rand Drop: {np.mean(rnd_drops):.4f}")
    print(f"  --> Paired t-test p-value:       {p_val_t:.4e}")
    print(f"  --> Wilcoxon signed-rank p-value: {p_val_w:.4e}")
    
    # Significance threshold check
    alpha = 0.05
    if p_val_w < alpha:
        print(f"  Conclusion: NMF drops are SIGNIFICANTLY greater than random drops (p < {alpha}).\n")
    else:
        print(f"  Conclusion: Not a statistically significant difference at alpha = {alpha}.\n")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. Calculate means and standard deviations (error bars) for each fraction
nmf_target_mean = [np.mean(all_nmf_target_drops[f]) for f in ABLATION_FRACTIONS]
nmf_target_err  = [np.std(all_nmf_target_drops[f]) for f in ABLATION_FRACTIONS]

nmf_byst_mean   = [np.mean(all_nmf_bystander_drops[f]) for f in ABLATION_FRACTIONS]
nmf_byst_err    = [np.std(all_nmf_bystander_drops[f]) for f in ABLATION_FRACTIONS]

rnd_target_mean = [np.mean(all_rnd_target_drops[f]) for f in ABLATION_FRACTIONS]
rnd_target_err  = [np.std(all_rnd_target_drops[f]) for f in ABLATION_FRACTIONS]

rnd_byst_mean   = [np.mean(all_rnd_bystander_drops[f]) for f in ABLATION_FRACTIONS]
rnd_byst_err    = [np.std(all_rnd_bystander_drops[f]) for f in ABLATION_FRACTIONS]

# 2. Set up the bar chart
x_pos = np.arange(len(ABLATION_FRACTIONS))
w = 0.18
fig, ax = plt.subplots(figsize=(9, 5))

# Plot bars with error bars (yerr) and caps (capsize)
ax.bar(x_pos - 1.5*w, nmf_target_mean, w, yerr=nmf_target_err, capsize=3, 
       label='NMF target', color='#c0392b', alpha=0.9, ecolor='black')
       
ax.bar(x_pos - 0.5*w, nmf_byst_mean, w, yerr=nmf_byst_err, capsize=3, 
       label='NMF bystander (mean)', color='#e74c3c', alpha=0.5, ecolor='black')
       
ax.bar(x_pos + 0.5*w, rnd_target_mean, w, yerr=rnd_target_err, capsize=3, 
       label='Random target', color='#2c3e50', alpha=0.9, ecolor='black')
       
ax.bar(x_pos + 1.5*w, rnd_byst_mean, w, yerr=rnd_byst_err, capsize=3, 
       label='Random bystander (mean)', color='#7f8c8d', alpha=0.5, ecolor='black')

# 3. Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{f:.0%}' for f in ABLATION_FRACTIONS])
ax.set_xlabel('Ablation fraction (top weights zeroed)')
ax.set_ylabel('Mean Accuracy Drop from Baseline')
ax.legend(fontsize=9, loc='upper left')
ax.set_title(f'Joint Ablation Performance Across All Paths (Mean ± Std)', fontsize=12)

# Optional: Add gridlines for easier reading
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True) 

plt.tight_layout()

# Save the aggregate figure (adjust directory if needed)
try:
    fig.savefig(os.path.join(FIG_DIR, 'faithfulness_joint_all_paths.png'), dpi=150, bbox_inches='tight')
except NameError:
    fig.savefig('faithfulness_joint_all_paths.png', dpi=150, bbox_inches='tight')

plt.show()

## Section 8: Deviations from the Reference Setup (NB02)

NB02 (`02_factor_trace.ipynb`) applies the NMF factor trace to a simple 3-layer MLP
on MNIST. NB07 extends this to CIFAR-10 with a convolutional network (SmallCNN).
Three structural deviations are required:

### 1. Conv Spatial Pooling (-> Section 2b)
NB02 has only FC layers; every weight connection has a unique scalar product with its
input neuron. For conv layers, each weight is replicated across all spatial positions.
Without pooling, the arbor matrix is ~1,000x too large. **Spatial pooling (avg/max/center)
collapses the spatial dimension**, producing a single per-kernel-entry summary.
This is a lossy operation: we lose spatial specificity in exchange for tractability.

### 2. Stimulus Filter (-> Section 2a)
NB02 uses all ~10,000 MNIST test images. CIFAR-10's activations are larger (32x32 color),
and the arbor matrices for conv layers scale as O(N x C_out x C_in x kH x kW).
**Keeping only N_TOP_PER_CLASS samples per class** bounds RAM usage while preserving
class representation. The filter (confidence or random) is validated in NB06 S4.

### 3. Auto-K NMF (-> Section 5)
NB02 used a fixed K for all layers. Here we fit NMF at K_MAX per layer and select K*
via the 'structural' method (fraction + cumvar heuristic). This respects that different
layers may need different numbers of components. Lambda plots during the trace show
the rank selection visually.
